# CGHNet — tái lập bài báo 0.818 trên test-104

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

Nguồn: Li, Sailike, Li, Zhang, Shang, Chen, *CGHNet: Cross-Guided 2D–3D Hybrid Network with
attention mechanism for focal liver lesion classification*, **Comput Med Imaging Graph 132
(2026) 102780**, doi:10.1016/j.compmedimag.2026.102780.

Cần mount **cache CGHNet** (sinh bằng `notebooks/18_build_cache_cghnet.ipynb`). Không cần
dữ liệu gốc, không cần internet.

## ⚠️ Đây là tái lập TỪ VĂN BẢN, không phải chạy lại code của tác giả

Bài **không công khai code**. Năm nhóm siêu tham số phải suy lại, xem `configs/cghnet.yaml`
— mọi khoá ở đó có nhãn `[BÀI]` hoặc `[SUY]`. **Không được lẫn hai nhãn khi viết báo cáo.**

Cổng A in số tham số thật cạnh **59.37M** của bài. Lệch nhiều nghĩa là ta dựng một kiến
trúc khác, và mọi so sánh với 0.818 phải nói rõ điều đó.

## Thang bậc chẩn đoán — lý do phép tái lập này well-posed

Bài train bằng multi-head deep supervision, `L = FL(ŷ) + FL(ŷ_2D) + FL(ŷ_3D)` (Eq. 12). Nên
**một lần chạy cho ba con số**, và cả ba có mốc công bố (Bảng 2, test-104):

| đầu ra | mốc | nếu lệch |
|---|---|---|
| nhánh **3D** một mình | **0.724** | xuống ~0.62 ⇒ sai **protocol/dữ liệu**, không phải fusion |
| nhánh **2D** một mình | **0.742** | lệch nhiều ⇒ sai nhánh ViT |
| **hợp nhất** | **0.818** | hai nhánh đúng mà cái này thấp ⇒ sai CGFM/ADF |

Thang bậc này **không tốn thêm giờ GPU nào**, và nó bao trọn phép thử "hình học 14×112×112
có phải nút thắt không". Mục 5 đọc nó ra từ checkpoint.

## Hai cảnh báo bắt buộc khi so với mốc của họ

⚠️ **Mốc của họ đo trên test-104; số ta có ngay là out-of-fold.** Thiên lệch chọn epoch của
dự án đã đo là **−0.069**, nên out-of-fold ~0.79 mới tương ứng test-104 ~0.72. Không so
trực tiếp.

⚠️ **Họ báo mean ± std của 5 model đơn**, không phải ensemble. Số so được của ta trên
test-104 là **0.6001**, không phải 0.6162.

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1, 2]                 # sàng 2 fold; đủ 5 fold mới kết luận con số
CONFIG_NAME = "cghnet.yaml"
PREPROCESS_NAME = "preprocess_cghnet.yaml"
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

EXPERIMENT = Path(CONFIG_NAME).stem
os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / CONFIG_NAME
CFG = load_yaml(CFG_PATH)
PRE = load_yaml(REPO / "configs" / PREPROCESS_NAME)
M = CFG["model"]

INNER = tuple(PRE["target_size"])
MARGIN = tuple(PRE["crop_margin_voxels"])
GRID = tuple(s + 2 * m for s, m in zip(INNER, MARGIN))
assert tuple(CFG["data"]["crop_size"]) == INNER, "data.crop_size lệch target_size của cache"

print(f"\nthí nghiệm: {EXPERIMENT} · fold {FOLDS}")
print(f"nhánh 2D  : ViT dim {M['embed_dim']} · depth {M['depth']} · {M['num_heads']} head · "
      f"patch {M['patch_size']}")
print(f"nhánh 3D  : resnet{M['resnet_depth']} -> token {M['token_dim']} · "
      f"conv1_stride {M['conv1_stride']}")
print(f"ADF       : lambda_res {M['lambda_res']}")
print(f"loss      : {CFG['loss']['name']} gamma {CFG['loss']['gamma']} · "
      f"deep_supervision {CFG['loss']['deep_supervision']}")
print(f"hình học  : cache {GRID} -> model nhận {INNER}")

## Cổng 0 ⚠️ — config có khớp đặc tả của bài không

CGHNet **cố ý** khác baseline ở nhiều khối: `model.`, `loss.`, `train.weight_decay`,
`data.batch_size`, `data.crop_size`, `data.augment`. Nên cổng này **không** assert "chỉ khác
một khối" như các thí nghiệm trước — khác nhiều ở đây là đúng đặc tả. Nó đối chiếu từng khoá
có nhãn `[BÀI]` với văn bản.

Đáng chú ý: **`weight_decay: 1e-5`**, khác baseline official (0.05) tới 5000 lần. Bài ghi
vậy ở §4.3, không phải sơ suất.

In [ ]:
BASE = load_yaml(REPO / "configs" / "baseline_3dpatch.yaml")


def flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flatten(v, prefix + k + "."))
        else:
            out[prefix + k] = v
    return out


fa, fb = flatten(BASE), flatten(CFG)
diff = sorted(k for k in set(fa) | set(fb) if str(fa.get(k)) != str(fb.get(k)))
print(f"{CONFIG_NAME} khác baseline ở {len(diff)} khoá:")
for k in diff:
    print(f"  {k}: {fa.get(k)!r} -> {fb.get(k)!r}")

# Đặc tả trích trực tiếp từ bài. Đây là cổng thật, không phải in cho đẹp.
DAC_TA_BAI = {
    "train.epochs": 300,                 # §4.3 "spanned 300 epochs"
    "train.lr": 0.0001,                  # §4.3 + Bảng 4 (1e-4 thắng 1e-3 và 1e-5)
    "train.weight_decay": 0.00001,       # §4.3 "weight decay of 1e-5"
    "train.warmup_epochs": 5,            # §4.3 "linear warm-up for the first 5 epochs"
    "loss.name": "focal",                # Bảng 4: focal 81.8 so với CE 79.9
    "loss.aux_weight": 1.0,              # Eq. 12 cộng không trọng số
    "model.lambda_res": 0.50,            # Bảng 6
    "data.augment.flip_prob": 0.5,       # §4.3 "each with a probability of 0.5"
    "data.augment.rotate_prob": 0.5,     # cùng câu trên — áp cho CẢ xoay
}
lech = {k: (v, fb.get(k)) for k, v in DAC_TA_BAI.items() if fb.get(k) != v}
assert not lech, f"⛔ lệch đặc tả của bài: {lech}"

hieu_dung = CFG["data"]["batch_size"] * CFG["train"]["accum_steps"]
assert hieu_dung == 4, f"§4.3 nói batch size 4, config cho hiệu dụng {hieu_dung}"
assert CFG["data"]["augment"]["rotate_mode"] == "nearest", (
    "lề cache chỉ 8 voxel mà xoay 10 độ hỏng góc ~12 voxel; `constant` để lọt voxel đệm 0"
)
print("\n✓ mọi khoá [BÀI] khớp văn bản · batch hiệu dụng 4 · rotate_mode nearest")

## 1. Cache CGHNet

Nhận diện bằng **nội dung** `cache_meta.json`, không bằng tên dataset.

⚠️ **Ba cache của dự án đều `per_phase` + `lesion_tight`**, nên hai khoá đó không phân biệt
được. Chỉ `target_size` + `crop_margin_voxels` phân biệt:

| | target_size | lề | mảng thật |
|---|---|---|---|
| E4 | [112,112,32] | none | (8,112,112,32) |
| E12 | [112,112,32] | [12,12,4] | (8,136,136,40) |
| **CGHNet** | **[112,112,14]** | **[8,8,1]** | **(8,128,128,16)** |

Cho nhầm cache thì model nhận hình học khác mà **không có gì báo lỗi**.

In [ ]:
import json as _json

import numpy as np

CAN = {
    "align_phases": "per_phase",
    "crop_mode": "lesion_tight",
    "target_size": list(INNER),
    "crop_margin_voxels": list(MARGIN),
}

ung_vien, CACHE_DIR = [], None
for meta_path in sorted(Path("/kaggle/input").rglob("cache_meta.json")):
    try:
        meta = _json.loads(meta_path.read_text("utf-8"))
    except Exception:  # noqa: BLE001 - chỉ để liệt kê chẩn đoán
        continue
    khop = all(meta.get(k) == v for k, v in CAN.items())
    ung_vien.append((meta_path.parent, meta, khop))
    if khop and CACHE_DIR is None:
        CACHE_DIR = meta_path.parent

print(f"=== {len(ung_vien)} cache tìm thấy dưới /kaggle/input ===")
for path, meta, khop in ung_vien:
    print(f"  {'✓ CGHNet' if khop else '  --    '}  {path}")
    print(f"          size={meta.get('target_size')} lề={meta.get('crop_margin_voxels')} "
          f"align={meta.get('align_phases')} crop={meta.get('crop_mode')}")

if CACHE_DIR is None:
    raise RuntimeError(
        "Chưa mount cache CGHNet.\n"
        f"  Cần cache có {CAN}\n"
        "  Chạy notebooks/18_build_cache_cghnet.ipynb trước (CPU, Accelerator = None,\n"
        "  ~20 phút), lưu output thành Dataset, rồi mount vào đây.\n"
        "  Cache E4 và cache E12 KHÔNG dùng được: hình học khác hẳn."
    )

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

meta = _json.loads((CACHE_DIR / "cache_meta.json").read_text("utf-8"))
n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498"
mau = next(CACHE_DIR.glob("*.npz"))
with np.load(mau) as z:
    shape = tuple(z["image"].shape)
assert shape == (8, *GRID), f"mảng {shape}, cần {(8, *GRID)} — đây không phải cache CGHNet"
print(f"\ncache ✓ · {CACHE_DIR} · {n_npz} ca · mảng {shape}")
print(f"  commit build: {meta.get('git_commit')}")

## Cổng A ⚠️⚠️ — kiến trúc dựng ra có đúng như bài không

Ba phép kiểm:

1. **Số tham số** cạnh 59.37M của bài (Bảng 5). Đây là bằng chứng gián tiếp duy nhất ta có
   về những siêu tham số bài không nói.
2. **Ba đầu ra tồn tại** và hợp đồng train/eval đúng: `train()` → dict, `eval()` → tensor.
   Cả `src/eval/*` dựa vào điều này.
3. **`λ_res` vào đúng công thức** Eq. 11: đặt `λ_res = 0` phải làm logit chính bằng đúng
   `ŷ_fus`. Nếu không thì hiệu chỉnh dư đang được nối sai chỗ.

In [ ]:
import torch

from src.models import build_model, count_parameters
from src.models.cghnet import PAPER_F1, PAPER_PARAMS_M

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(CFG["model"]).to(dev)
n_param = count_parameters(model)
print(f"tham số: {n_param:,}  ({n_param / 1e6:.2f}M)")
print(f"bài báo: {PAPER_PARAMS_M:.2f}M   lệch {n_param / 1e6 - PAPER_PARAMS_M:+.2f}M "
      f"({(n_param / 1e6 / PAPER_PARAMS_M - 1) * 100:+.0f}%)")
if abs(n_param / 1e6 - PAPER_PARAMS_M) > 10:
    print("⚠ lệch quá 10M — ta đang dựng một kiến trúc khác đáng kể. Vẫn chạy được, nhưng")
    print("  mọi so sánh với 0.818 phải nói rõ điều đó.")

# Chạy trên GPU: ResNet-50-3D + ViT trên 4 vCPU thì chờ khá lâu.
x_probe = torch.zeros(2, CFG["model"]["num_phases"], *INNER, device=dev)

model.train()
out = model(x_probe)
assert isinstance(out, dict) and set(out) == {"main", "aux"}, "train mode phải trả dict"
assert set(out["aux"]) == {"2d", "3d"}
for ten, logits in [("main", out["main"]), ("2d", out["aux"]["2d"]), ("3d", out["aux"]["3d"])]:
    assert tuple(logits.shape) == (2, 7), (ten, logits.shape)
print(f"\nba đầu ra ✓  main {tuple(out['main'].shape)} · "
      f"2d {tuple(out['aux']['2d'].shape)} · 3d {tuple(out['aux']['3d'].shape)}")

model.eval()
with torch.no_grad():
    tensor_out = model(x_probe)
assert torch.is_tensor(tensor_out), "eval mode phải trả TENSOR (src/eval/* dựa vào điều này)"
print(f"eval mode trả tensor ✓  {tuple(tensor_out.shape)}")

# lambda_res = 0 => logit chính phải bằng đúng y_fus. Kiểm bằng cách so với đường tính tay.
with torch.no_grad():
    heads = model.forward_heads(x_probe)
    beta = model.last_beta
    tay = heads["main"] - M["lambda_res"] * (
        beta * heads["aux"]["2d"] + (1.0 - beta) * heads["aux"]["3d"]
    )
    model.lambda_res = 0.0
    y_fus = model.forward_heads(x_probe)["main"]
    model.lambda_res = float(M["lambda_res"])
torch.testing.assert_close(tay, y_fus, rtol=1e-4, atol=1e-4)
print(f"lambda_res={M['lambda_res']} vào đúng Eq. 11 ✓")
print(f"\nmốc công bố sẽ đối chiếu ở mục 5: {PAPER_F1}")

## Cổng B ⚠️⚠️⚠️ — hình dạng qua từng khối

Cell này gắn forward hook và **đo hình dạng thật** thay vì tin config. Đây là bản CGHNet
của cổng đã bắt lỗi E2 (E2 chạy ở 48×48 in-plane suốt 100+ epoch và không có gì báo).

Kỳ vọng với đầu vào `[B, 8, 112, 112, 14]`:

| | hình dạng |
|---|---|
| chuỗi nhánh 2D | `[B, 14, 384]` — **một token mỗi LÁT** |
| feature map `layer4` | `[B, 2048, 7, 7, 1]` |
| chuỗi nhánh 3D | `[B, 49, 384]` |
| trọng số lát / voxel | `[B, 14]` / `[B, 49]`, mỗi cái tổng = 1 |

⚠️ Trục z co về **1** ở `layer4` là **tất yếu** của z=14 với 16 lần hạ mẫu, không phải lỗi.
Có test riêng neo lại điều này để lần sau không ai đi "sửa".

In [ ]:
from src.train.run import build_loaders

train_loader, val_loader, _ = build_loaders(CFG, FOLDS[0])
batch = next(iter(train_loader))
x = batch["image"]
B = x.shape[0]
print(f"batch từ loader: {tuple(x.shape)}")
assert tuple(x.shape[2:]) == INNER, (
    f"⛔ loader cho {tuple(x.shape[2:])} thay vì {INNER} — cắt ngẫu nhiên không chạy"
)

model = model.eval()   # `dev` và model đã lên GPU ở cổng A

seen = {}
h1 = model.blocks[-1].register_forward_hook(
    lambda m, i, o: seen.update(vit_token=tuple(o.shape))
)
h2 = model.resnet.layer4.register_forward_hook(lambda m, i, o: seen.update(map3d=tuple(o.shape)))
h3 = model.cgfm.register_forward_hook(
    lambda m, i, o: seen.update(z2d=tuple(o[0].shape), z3d=tuple(o[1].shape))
)
with torch.no_grad():
    heads = model.forward_heads(x.to(dev))
for h in (h1, h2, h3):
    h.remove()

print(f"token ViT (gộp B*Z) : {seen['vit_token']}")
print(f"feature map layer4  : {seen['map3d']}")
print(f"chuỗi 2D sau CGFM   : {seen['z2d']}")
print(f"chuỗi 3D sau CGFM   : {seen['z3d']}")
print(f"logits              : {tuple(heads['main'].shape)}")

n_slice = INNER[2]
assert seen["z2d"] == (B, n_slice, M["embed_dim"]), (
    f"⛔ chuỗi 2D là {seen['z2d']}, cần {(B, n_slice, M['embed_dim'])} — một token mỗi LÁT"
)
assert seen["map3d"][2:] == (7, 7, 1), f"vết không gian {seen['map3d'][2:]}, mong đợi (7,7,1)"
n_v = seen["map3d"][2] * seen["map3d"][3] * seen["map3d"][4]
assert seen["z3d"] == (B, n_v, M["token_dim"]), f"chuỗi 3D {seen['z3d']}, cần N_v={n_v}"

ws, wv = model.last_slice_weights.cpu(), model.last_voxel_weights.cpu()
assert tuple(ws.shape) == (B, n_slice) and tuple(wv.shape) == (B, n_v)
torch.testing.assert_close(ws.sum(dim=1), torch.ones(B))
torch.testing.assert_close(wv.sum(dim=1), torch.ones(B))
print(f"\n✓ hình học đúng đặc tả · N_v = {n_v} · trọng số attention chuẩn hoá")

## Cổng C ⚠️ — ngân sách

Bài train trên **RTX 4090**; T4 của Kaggle chậm hơn nhiều và chỉ đạt ~2 TFLOPS hiệu dụng
trên conv 3D (đo được ở E8 và E13).

⚠️ **Đừng dùng con số 209.91 GFLOPs của bài để suy giờ.** Nó lệch đáng ngờ so với 31.31
GFLOPs của `ResNet3D` cùng bảng, và ta không biết họ đo ở kích thước nào. Đo thật.

| GPU đo được | làm gì |
|---|---|
| dưới ~45 s/epoch | CPU chặn, ~3,8 h/fold. Hai fold lọt một session |
| 45–65 s/epoch | GPU chặn, ~4–5,5 h/fold. Hai fold vẫn lọt |
| trên ~65 s/epoch | fold 1 ở session này, fold 2 ở session sau (`resume: true` nên không mất gì). Hoặc đặt `model.conv1_stride: [2, 2, 1]` cho nhánh 3D rẻ hơn 4 lần |

In [ ]:
import time

model = model.train()
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda")

from src.train.losses import build_criterion

criterion = build_criterion(CFG, [0, 1, 2, 3, 4, 5, 6], dev)
xb = x.to(dev)
yb = batch["label"].to(dev)


def buoc():
    opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16):
        out = model(xb)
        loss = criterion(out, yb)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    return out, loss


for _ in range(3):
    out, loss = buoc()
torch.cuda.synchronize()
t0 = time.time()
for _ in range(12):
    buoc()
torch.cuda.synchronize()
giay_batch = (time.time() - t0) / 12

so_batch = len(train_loader)
t0, n = time.time(), 0
for _ in train_loader:
    n += 1
    if n >= 15:
        break
cpu_epoch = (time.time() - t0) / n * so_batch
gpu_epoch = giay_batch * so_batch
epoch = max(cpu_epoch, gpu_epoch)
gio_fold = epoch * int(CFG["train"]["epochs"]) / 3600

print(f"VRAM đỉnh   : {torch.cuda.max_memory_allocated() / 2**30:.1f} GB "
      f"(bài: 2.28 GB ở batch 1 inference)")
print(f"GPU fwd+bwd : {gpu_epoch:6.0f} s/epoch  ({so_batch} batch × {giay_batch:.3f}s)")
print(f"CPU nạp+aug : {cpu_epoch:6.0f} s/epoch")
print(f"ước tính    : {epoch:6.0f} s/epoch -> {gio_fold:.1f} h/fold -> "
      f"{gio_fold * len(FOLDS):.1f} h cho {len(FOLDS)} fold")
if gio_fold * len(FOLDS) > 11.0:
    print(f"\n⚠ {gio_fold * len(FOLDS):.1f}h vượt trần session 12h — chạy FOLDS = [{FOLDS[0]}]")
    print("  ở session này, fold còn lại ở session sau. `resume: true` nên không mất gì.")

# --- Cổng D: deep supervision có sống không ---
# Nếu một đầu phụ không nhận gradient thì thang bậc chẩn đoán biến mất mà không có gì báo.
grads = {}
for ten, head in (("2d", model.head_2d), ("3d", model.head_3d), ("fus", model.head_fus)):
    w = head[1].weight
    grads[ten] = float(w.grad.abs().sum()) if w.grad is not None else 0.0
print(f"\n|grad| của ba đầu: " + " · ".join(f"{k} {v:.3e}" for k, v in grads.items()))
assert all(v > 0 for v in grads.values()), (
    f"⛔ đầu {[k for k, v in grads.items() if v == 0]} KHÔNG nhận gradient — deep "
    "supervision không sống, và thang bậc chẩn đoán 0.724/0.742/0.818 mất hiệu lực"
)
print("✓ cả ba đầu đều nhận gradient")

del opt, xb, yb
torch.cuda.empty_cache()

## 2. Train

`resume: true` nên bị ngắt giữa chừng thì chạy lại đúng cell này, nó đọc tiếp từ `last.pt`.

⚠️ `train` đọc YAML **từ đĩa**, không dùng biến `CFG`. Sửa `CFG` bằng tay ở cell nào đó sẽ
không có tác dụng lúc train, mà cổng 0 lại đọc từ chính file nên vẫn báo xanh.

### Cái gì hiện ra mỗi epoch

Hai dòng log, đi qua `logger.info` nên hiện sạch ở cả chạy tương tác lẫn `Save & Run All`:

    epoch 12/300 | train 1.8016 | val 1.9433 | macro-F1 0.3120 | 96s
            F1: u máu 0.615 · ICC 0.000 · áp-xe 0.222 · di căn 0.000 · nang 0.667 · FNH 0.000 · HCC 0.481

**Đọc dòng F1 từng lớp quan trọng ngang macro-F1.** Giữ nguyên ICC 0.519 và di căn 0.273 thì
kể cả 5 lớp kia đều đạt 0.90, macro-F1 cũng chỉ tới 0.756 — nên hai lớp đó là thứ chặn mục
tiêu về mặt số học, mà macro-F1 gộp lại che mất chúng. F1 từng lớp cũng vào `train_log.csv`
thành cột riêng (`f1_ICC`, `f1_di căn`, ...) nên vẽ được quỹ đạo theo epoch ở local.

⚠️ **ICC và di căn bằng 0 trong vài chục epoch đầu là bình thường** — model chưa học được lớp
hiếm. Đáng lo là khi chúng vẫn 0 sau epoch ~100.

**Không có thanh tiến độ trong epoch, và đó là chủ ý** (WORKLOG S-122). Bản tqdm đã dựng rồi
bỏ: ở `Save & Run All` nó vô dụng theo cả hai nhánh — bản widget không có frontend nào nhận
cập nhật, còn bản text thì log lưu lại không gộp `
` nên thành hàng nghìn dòng lặp. Với
~96 s/epoch thì dòng log mỗi epoch đã là nhịp phản hồi đủ dày.

In [ ]:
from src.train.run import train

t0 = time.time()
results = {}

for fold in FOLDS:
    print("\n" + "=" * 60)
    print("FOLD %d  (da dung %.2fh)" % (fold, (time.time() - t0) / 3600))
    print("=" * 60)
    results[fold] = train(CFG_PATH, fold_override=fold)
    print("fold %d xong: macro-F1 %.4f" % (fold, results[fold]["macro_f1"]))

print("\ntong: %.2f h" % ((time.time() - t0) / 3600))

## 3. Kết quả từng fold

In [ ]:
import csv as _csv

OUT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
E4 = {1: 0.7001, 2: 0.6771, 3: 0.7304, 4: 0.6680, 5: 0.6618}
E4_N = {1: 82, 2: 80, 3: 78, 4: 77, 5: 77}

print(f"{'fold':>5}{'epoch':>7}{'macro-F1':>11}{'kappa':>9}{'E4':>9}{'hiệu':>9}")
print("-" * 50)
rows, tong_n, tong_f1, tong_e4 = [], 0, 0.0, 0.0
for d in sorted(OUT.glob("fold*")):
    f = d / "metrics_best.json"
    if not f.exists():
        continue
    m = _json.loads(f.read_text("utf-8"))
    fold = int(m["fold"])
    rows.append((d, fold, m))
    n = E4_N[fold]
    tong_n += n
    tong_f1 += n * m["macro_f1"]
    tong_e4 += n * E4[fold]
    print(f"{fold:>5}{m['epoch']:>7}{m['macro_f1']:>11.4f}{m['cohen_kappa']:>9.4f}"
          f"{E4[fold]:>9.4f}{m['macro_f1'] - E4[fold]:>+9.4f}")

if tong_n:
    tb, tb_e4 = tong_f1 / tong_n, tong_e4 / tong_n
    print("-" * 50)
    print(f"{'TB':>5}{'':>7}{tb:>11.4f}{'':>9}{tb_e4:>9.4f}{tb - tb_e4:>+9.4f}")

print("\nEpoch chạm đáy val_loss (ρ=0.770 với F1 cuối, S-107):")
for d, fold, _m in rows:
    log = d / "train_log.csv"
    if log.exists():
        vl = [float(r["val_loss"]) for r in _csv.DictReader(open(log))]
        print(f"  fold {fold}: đáy @ epoch {vl.index(min(vl)) + 1}")

print("\n⚠ Chênh lệch từng fold là nhiễu nếu nhìn riêng (CI mỗi fold ~±0.19).")

## 4. Thang bậc chẩn đoán ⚠️ — mục quan trọng nhất của notebook

Đọc **ba** đầu ra từ checkpoint `best.pt` bằng `forward_heads()`, rồi đối chiếu với mốc
công bố. Đây là chỗ phân biệt được ba nguyên nhân thất bại khác nhau, và nó chỉ tốn ~1 phút.

Dùng `forward_heads()` ở chế độ **eval** chứ không bật `train()`: bật train sẽ kéo
BatchNorm sang thống kê của batch hiện tại và số ra không còn là số của checkpoint.

⚠️ Số dưới đây là **out-of-fold**, mốc của họ là **test-104**. Thiên lệch chọn epoch của dự
án là −0.069, nên cột "tương ứng test-104" là số out-of-fold trừ đi 0.069.

In [ ]:
from src.eval.metrics import macro_f1

BIAS = 0.069   # thiên lệch chọn epoch đã đo trên out-of-fold của dự án (S-110)

for d, fold, _m in rows:
    ckpt = torch.load(d / "best.pt", map_location="cpu")
    net = build_model(CFG["model"])
    net.load_state_dict(ckpt["model"])
    net = net.to(dev).eval()

    _, val_d, _ = build_loaders(CFG, fold)
    probs = {k: [] for k in ("main", "2d", "3d")}
    labels = []
    with torch.no_grad():
        for b in val_d:
            heads = net.forward_heads(b["image"].to(dev))
            probs["main"].append(torch.softmax(heads["main"].float(), 1).cpu().numpy())
            probs["2d"].append(torch.softmax(heads["aux"]["2d"].float(), 1).cpu().numpy())
            probs["3d"].append(torch.softmax(heads["aux"]["3d"].float(), 1).cpu().numpy())
            labels.append(b["label"].numpy())
    y = np.concatenate(labels)

    print(f"\n=== FOLD {fold} · epoch {ckpt['epoch']} · {len(y)} ca val ===")
    print(f"{'đầu ra':<22}{'out-of-fold':>13}{'~test-104':>12}{'mốc bài':>10}{'hiệu':>9}")
    print("-" * 66)
    for key, ten in (("3d", "nhánh 3D"), ("2d", "nhánh 2D"), ("main", "hợp nhất (CGHNet)")):
        p = np.concatenate(probs[key])
        f1 = macro_f1(y, p.argmax(1))
        uoc = f1 - BIAS
        moc = PAPER_F1[key]
        print(f"{ten:<22}{f1:>13.4f}{uoc:>12.4f}{moc:>10.3f}{uoc - moc:>+9.4f}")

    # Lưu lại để đọc ở local mà không phải chạy lại GPU.
    np.savez_compressed(
        d / "val_probs_best_heads.npz",
        main=np.concatenate(probs["main"]),
        aux2d=np.concatenate(probs["2d"]),
        aux3d=np.concatenate(probs["3d"]),
        labels=y,
        epoch=ckpt["epoch"],
    )
    del net
    torch.cuda.empty_cache()

print("""
CÁCH ĐỌC — theo đúng thứ tự này, đừng nhảy bước:

  1. nhánh 3D (~test-104) thấp hơn 0.66  ->  SAI PROTOCOL/DỮ LIỆU, không phải sai fusion.
     Dừng, soi cổng B và cache. Mọi số khác chưa đáng tin.
  2. nhánh 3D quanh 0.72 mà nhánh 2D thấp hẳn  ->  sai nhánh ViT.
  3. hai nhánh đều quanh mốc mà hợp nhất không hơn  ->  sai CGFM/ADF.
  4. cả ba quanh mốc  ->  bản tái lập đứng vững; chạy đủ 5 fold rồi mới bàn test-104.

⚠️ Nhánh 3D đạt ~0.72 là PHÁT HIỆN QUAN TRỌNG NHẤT có thể có ở đây, kể cả khi hợp nhất
   không tới 0.818: nó nghĩa là hình học 14x112x112 giải thích phần lớn khoảng cách 0.109,
   và điều đó áp cho CẢ E4 lẫn E13 — rẻ hơn nhiều so với xây thêm kiến trúc.
""")

## 5. Gói mang về

In [ ]:
import shutil

PACK = Path(f"/kaggle/working/{EXPERIMENT}_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

KEEP = ["val_probs_best.npz", "val_probs_last.npz", "val_probs_best_heads.npz",
        "metrics_best.json", "train_log.csv", "config_used.json", "best.pt"]
for d in sorted(OUT.glob("fold*")):
    dst = PACK / d.name
    dst.mkdir(parents=True, exist_ok=True)
    for name in KEEP:
        if (d / name).exists():
            shutil.copy2(d / name, dst / name)

total = sum(f.stat().st_size for f in PACK.rglob("*") if f.is_file())
print(f"đã gói {PACK}: {total / 2**20:.1f} MiB")
print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz và .pt bản thân là zip; trình giải nén bung
  đệ quy sẽ biến chúng thành thư mục và src.eval.* không thấy (đã dính, S-078).

Ở local, đặt vào runs/CGHNET/ rồi:
    python -m src.eval.compare --baseline runs/E4_cv_results --candidate runs/CGHNET
    python -m src.eval.run     --run-dir runs/CGHNET

`val_probs_best_heads.npz` giữ cả ba đầu ra, nên đọc lại thang bậc ở local được, không
phải chạy lại GPU.
""")